In [1]:
# auto reload 
%load_ext autoreload
%autoreload 2

In [2]:
import os

In [3]:
print(os.getcwd())

/home/nadav/dev/PhoCoLens/SVDeconv


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sacred import Experiment
import cv2
import torch

In [ ]:
from models.fftlayer_diff import FFTLayer_diff,load_psf
from config_diffusercam import initialise
from utils.tupperware import tupperware
from dataloader import get_dataloaders
from utils.ops import rggb_2_rgb, unpixel_shuffle
from utils.model_serialization import load_state_dict



In [9]:
from argparse import Namespace

In [6]:
from SRIL_utils.plot_utils import myim,plot_image_histogram,plot_hist

In [10]:
from models.unet import UNet270480 as Unet_diff
pixelshuffle_ratio = 2
in_c = 3
args = Namespace(pixelshuffle_ratio=2)

In [11]:
args 

Namespace(pixelshuffle_ratio=2)

In [30]:
unet = Unet_diff(args, in_c=in_c)

In [46]:
270/480,720/1280


(0.5625, 0.5625)

In [47]:
temp = torch.rand(4,3,720, 1280)

In [19]:
temp_unpixel_shuffled = unpixel_shuffle(
                    temp, args.pixelshuffle_ratio
                )

In [48]:
torch_pixel_unshuffle = torch.nn.PixelUnshuffle(downscale_factor=args.pixelshuffle_ratio)
torch_pixel_shuffle = torch.nn.PixelShuffle(upscale_factor=args.pixelshuffle_ratio)

In [49]:
#torch.min(unpixel_shuffle(temp, args.pixelshuffle_ratio,) - toch_unpixel_shuffle(temp))

In [50]:
temp_unpixel_shuffled=  torch_pixel_unshuffle(temp)

In [51]:
unet_out_shuffeled  = unet(temp_unpixel_shuffled)

/home/nadav/dev/PhoCoLens/SVDeconv/models/unet.py:50: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  x = F.upsample(x, size=(height, width), mode='bilinear')


In [52]:
unet_out_shuffeled.shape

torch.Size([4, 12, 360, 640])

In [53]:
unet_out = torch_pixel_shuffle(unet_out_shuffeled)

In [54]:
unet_out.shape

torch.Size([4, 3, 720, 1280])